# A1 worker (Kaggle)
Runs this person's share of the A1 experiment. Thin notebook: all logic is in `A1/a1lib` (see `A1/TEAM_PROTOCOL.md`).

**Before the first run** (Notebook settings, right side): Accelerator **GPU T4 x2**, Internet **On**, *Add Input* → the private Food-101 dataset, *Add-ons → Secrets* → `HF_TOKEN` (Hugging Face write token), *Session options* → Persistence: none is fine.

**Daily use:** set the cell 1 values, then *Run All* (interactive session) or *Save Version → Save & Run All (Commit)* (runs unattended, keeps going when you close the tab). Stopping and restarting is always safe: runs resume from the store.

**More than one session:** open the same notebook in a second session (or a second copy) and run it too. Workers coordinate through locks in the store; see the protocol, section 5.

In [ ]:
# 1) your settings
OWNER = "dat"                         # "dat" or "ly"  (which runs are yours in plan.json)
GPUS = 2                              # one worker per GPU (T4 x2); use 1 for a single-GPU session
MAX_HOURS = 8.5                       # stop cleanly (checkpoint pushed) before Kaggle's 12 h limit; lower it if your weekly quota is nearly used
HF_REPO = "dl4dl/a1-store"            # TODO: the private Hugging Face repo shared by both members (organization/name)
REPO_URL = "https://github.com/broistg/deep-learning-CO5085.git"
BRANCH = "main"
STEAL = False                         # True: also take unfinished, unlocked runs of the other owner (agree in chat first)
RUN_SMOKE = False                     # True once, on a new account: 2 tiny epochs, nothing written to the store

In [ ]:
# 2) set up: code, libraries, secrets, environment (about 1-2 min; torch/torchvision are preinstalled and are not touched)
import os, subprocess, sys
REPO = "/kaggle/working/repo"
def sh(c): print("$", c); subprocess.run(c, shell=True, check=True)
if not os.path.isdir(REPO + "/.git"):
    sh(f"git clone -b {BRANCH} {REPO_URL} {REPO}")
else:
    sh(f"cd {REPO} && git pull --ff-only")
os.chdir(REPO)
sh("pip install -q -r requirements/a1.txt && pip install -q -e .")
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")      # never printed
os.environ.update(A1_STORE=f"hf:{HF_REPO}", A1_OWNER=OWNER, A1_LOCAL="/kaggle/temp/a1_local",
                  PYTHONPATH=f"{REPO}:{REPO}/A1", A1_PUSH_EVERY_MIN="45")
sh("git log -1 --format='commit %h %s'; nvidia-smi -L; df -h /kaggle/temp | tail -1")

In [ ]:
# 3) look at the store and the run list (read-only)
sh("python -m a1lib.store stats")
sh(f"python -m a1lib.worker status --a1-dir A1 | tail -25")

In [ ]:
# 4) optional smoke test (RUN_SMOKE = True): needs the dataset; copies it to local disk first (about 5 min), then trains 2 x 6 batches
if RUN_SMOKE:
    sh("python -m a1lib.worker prepare")
    sh(f"python -m a1lib.train --config A1/configs/m_resnet50_head_s0.yaml --smoke --owner {OWNER}")

In [ ]:
# 5) THE LONG CELL: copy the dataset once, then one worker per GPU. Output lines are tagged [g0] / [g1].
steal = "--steal" if STEAL else ""
sh(f"python -m a1lib.worker launch --owner {OWNER} --gpus {GPUS} --max-hours {MAX_HOURS} {steal}")

In [ ]:
# 6) after the session: where are we, and how full is the store history?
sh("python -m a1lib.worker status --a1-dir A1 | tail -25")
sh("python -m a1lib.store stats")